In [ ]:
# 200,000 rows: above the 100,000 threshold the grid holds no rows; Python answers each window.
import numpy as np
import pandas as pd
from lattice_grid_jupyter import LatticeGridWidget

n = 200_000
df = pd.DataFrame({
    "id": np.arange(n),
    "team": ["t%d" % (i % 4) for i in range(n)],
    "amount": (np.arange(n) % 100).astype("float64"),
})
w = LatticeGridWidget(df, offline=True, state={"version": 2, "group": ["team"]})
print("MODE=%s" % ("windowed" if w.windowed else "client"))

In [ ]:
w

In [ ]:
# Group by team (the state above), and filter to amount >= 50.
w.state = {"version": 2, "group": ["team"],
           "filters": {"col": "amount", "op": "gte", "value": 50}}

In [ ]:
# A later cell: the view's row count and the group aggregates the grid asks for.
view = w.view
sent = []
w.send = lambda content, buffers=None: sent.append(content)
w._on_custom(w, {"type": "lg:req", "id": 1, "method": "grouplevel", "payload": {
    "query": {"groupBy": ["team"], "groupValues": [], "sort": [],
              "filters": w.state["filters"], "range": {"start": 0, "end": 10}},
    "aggregates": [{"id": "total", "col": "amount", "fn": "sum"}]}}, [])
res = sent[0]
assert res["ok"], res
rows = res["result"]["rows"]
want = df[df["amount"] >= 50]
assert len(view) == len(want)
assert [r["team"] for r in rows] == ["t0", "t1", "t2", "t3"]
assert [r["leafCount"] for r in rows] == list(want.groupby("team").size())
print("LARGE_ROWS=%d" % len(view))
print("LARGE_GROUPS=%d" % len(rows))
print("LARGE_T0_COUNT=%d" % rows[0]["leafCount"])
print("LARGE_T0_TOTAL=%d" % rows[0]["totals"]["total"])
assert rows[0]["totals"]["total"] == want[want["team"] == "t0"]["amount"].sum()